# Zero-Shot Foundation Depth Models vs USS Ground Truth (Near-Field Parking)

**Purpose.** Test the hypothesis that modern monocular metric-depth foundation models
fail in the near-field parking regime (< 2–4 m, low/thin obstacles) where ultrasonic
sensors (USS) provide reliable distances. If the failure is systematic, it motivates the
paper's core claim: *USS can serve as free supervision for camera-only near-field
obstacle perception.*

**Protocol (v1, single sequence).**
1. Load one raw sequence (metadata JSON, Label V2 CSV, USS classifier-monitor log, camera video).
2. Select the evidence camera from driving direction (`backward -> Rear`, `forward -> Front`).
3. Extract frames at several time ratios along the approach.
4. Run Hugging Face metric-depth models zero-shot (Depth Anything V2 Metric, ZoeDepth, Depth Pro).
5. Compare predicted near-field depth against USS distances, and inspect whether thin
   structures (here: bicycle stands) appear in the depth maps at all.

**Ground truth available in this sequence.**
- Label V2 event label: object `bicyclestand`, severity class 3 (high), `important Distance = 2000 mm`.
- USS classifier monitor: per-cycle `ObjDist` in mm for tracked objects (an approach curve).

**Known caveats — read before interpreting results.**
- Camera intrinsics are unknown; metric models assume their training intrinsics
  (Depth Pro estimates focal length itself). Absolute scale errors are therefore expected —
  the interesting signals are *structure* (do thin obstacles exist in the depth map?) and
  *consistency* (does predicted distance track the USS approach curve?).
- USS cycle numbers are mapped to video time by a linear approximation (no shared clock
  parsed yet). Treat the time axis of the comparison as approximate.
- Label V2 / USS map coordinates are a *global map frame in millimeters* (~733,000 / ~1,183,000),
  not ego-relative. Do not overlay them on camera frames.


## 1. Environment setup

Works in Google Colab (mount Drive or clone the repo) and locally.

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if IN_COLAB:
    pip_install("transformers>=4.45", "accelerate")
# Local runs assume the repo environment (uv) already provides these.

import json
import lzma
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"colab={IN_COLAB} device={DEVICE}")

In [ ]:
SEQUENCE_NAME = "110613_20190308_LB_XO2617_039_151540"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    # EDIT: point at the sequence folder on your Drive.
    SEQUENCE_DIR = Path("/content/drive/MyDrive/vision-uss-research/data/samples") / SEQUENCE_NAME
    OUTPUT_ROOT = Path("/content/outputs/experiments/depth_zero_shot")
else:
    here = Path.cwd()
    repo_root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
    SEQUENCE_DIR = repo_root / "data" / "samples" / SEQUENCE_NAME
    OUTPUT_ROOT = repo_root / "outputs" / "experiments" / "depth_zero_shot"

OUTPUT_DIR = OUTPUT_ROOT / SEQUENCE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert SEQUENCE_DIR.is_dir(), f"sequence dir not found: {SEQUENCE_DIR}"
print(SEQUENCE_DIR)

## 2. Load sequence metadata and select the evidence camera

In [ ]:
seq_stem = SEQUENCE_NAME.split("_", 1)[1]  # drop the numeric id prefix

with open(SEQUENCE_DIR / f"{seq_stem}.json") as f:
    metadata = json.load(f)

tags = {}
for tag in metadata.get("tags", []):
    tags.setdefault(tag["tagSubCategory"]["name"], []).append(tag["name"])

driving_direction = tags.get("Driving Direction", ["unknown"])[0]
selected_camera = {"backward": "Rear", "forward": "Front"}.get(driving_direction)
assert selected_camera, f"unhandled driving direction: {driving_direction}"

video_path = SEQUENCE_DIR / f"{seq_stem}_{selected_camera}.avi.mp4"
assert video_path.exists(), video_path

print(f"driving direction : {driving_direction}")
print(f"selected camera   : {selected_camera}")
print(f"label objects     : {tags.get('Label Objects')}")
print(f"scene / weather   : {tags.get('Scene')} / {tags.get('Weather Conditions')}")
print(f"speed range kph   : {metadata.get('minSpeedInKph')}-{metadata.get('maxSpeedInKph')}")

In [ ]:
label_v2 = pd.read_csv(SEQUENCE_DIR / f"{seq_stem}_PAS_m_label_V2.csv", encoding="utf-8-sig")

CLASS_COL = "class (0=traversable, 1=low, 2=unknown, 3=high)"
DIST_COL = "important Distance [mm]"
SEVERITY_NAMES = {0: "traversable", 1: "low", 2: "unknown", 3: "high"}

label_rows = label_v2[["Which object", CLASS_COL, DIST_COL]].copy()
label_rows["severity"] = label_rows[CLASS_COL].map(SEVERITY_NAMES)

# The obstacle row (non-traversable) defines the labeled event distance.
obstacle_rows = label_rows[label_rows[CLASS_COL] > 0]
event_distance_m = float(obstacle_rows[DIST_COL].iloc[0]) / 1000.0
target_object = obstacle_rows["Which object"].iloc[0]

print(label_rows)
print(f"\ntarget object: {target_object!r}, labeled event distance: {event_distance_m:.2f} m")

## 3. USS ground-truth approach curve

`MAP_ClassifierMonitorNormed_ObjDist` gives a per-USS-cycle distance (mm) to each tracked
object. Cycle numbers are mapped linearly onto video time — an approximation until the
MF4 clock is parsed.

In [ ]:
cus_replay = next((SEQUENCE_DIR / "CusReplay").glob("*/*Classifier_Monitor_Normed.csv.xz"))

with lzma.open(cus_replay, "rt", errors="replace") as f:
    monitor = pd.read_csv(f, sep=";", low_memory=False)

OBJ_DIST = "MAP_ClassifierMonitorNormed_ObjDist"
OBJ_ID = "MAP_ClassifierMonitorNormed_ObjID"
uss = monitor[monitor[OBJ_DIST] > 0][["CycleNr", OBJ_ID, OBJ_DIST]].copy()
uss["dist_m"] = uss[OBJ_DIST] / 1000.0

cap = cv2.VideoCapture(str(video_path))
video_duration_s = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
cap.release()

c_min, c_max = monitor["CycleNr"].min(), monitor["CycleNr"].max()
uss["approx_t_s"] = (uss["CycleNr"] - c_min) / (c_max - c_min) * video_duration_s

fig, ax = plt.subplots(figsize=(10, 4))
for obj_id, grp in uss.groupby(OBJ_ID):
    ax.plot(grp["approx_t_s"], grp["dist_m"], marker=".", ms=3, lw=1, label=f"USS obj {obj_id}")
ax.axhline(event_distance_m, color="k", ls="--", label=f"Label V2 event ({event_distance_m:.1f} m)")
ax.set_xlabel("approx. video time [s]")
ax.set_ylabel("USS distance [m]")
ax.set_title(f"USS approach curves — {target_object}")
ax.legend(fontsize=8, ncol=3)
fig.savefig(OUTPUT_DIR / "uss_approach_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Nearest reported USS object at each moment = the safety-relevant distance.
uss_nearest = uss.groupby("CycleNr").agg(approx_t_s=("approx_t_s", "first"), dist_m=("dist_m", "min"))

## 4. Extract frames along the approach

In [ ]:
FRAME_RATIOS = [0.30, 0.50, 0.70, 0.85, 0.95, 0.99]

def extract_frame(path: Path, t_s: float) -> np.ndarray:
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_MSEC, t_s * 1000.0)
    ok, frame_bgr = cap.read()
    cap.release()
    assert ok, f"failed to read frame at t={t_s:.2f}s"
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

frames = {r: extract_frame(video_path, r * video_duration_s) for r in FRAME_RATIOS}

fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 3))
for ax, (r, img) in zip(axes, frames.items()):
    ax.imshow(img)
    ax.set_title(f"t={r * video_duration_s:.1f}s (ratio {r})", fontsize=9)
    ax.axis("off")
fig.suptitle(f"{selected_camera} camera — approach toward {target_object}")
fig.savefig(OUTPUT_DIR / "frames.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Foundation depth models (Hugging Face)

All models are run through the same `transformers` interface and return **metric depth in
meters** at the input resolution. Toggle models in `ENABLED_MODELS` — Depth Pro is the
strongest but needs a GPU runtime (~2 GB weights).

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

MODEL_ZOO = {
    "DA-V2-Metric-Outdoor-S": "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf",
    "DA-V2-Metric-Outdoor-L": "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf",
    "ZoeDepth-NK": "Intel/zoedepth-nyu-kitti",
    "DepthPro": "apple/DepthPro-hf",
}

ENABLED_MODELS = ["DA-V2-Metric-Outdoor-S", "ZoeDepth-NK"]
if DEVICE == "cuda":
    ENABLED_MODELS += ["DA-V2-Metric-Outdoor-L", "DepthPro"]

def load_depth_model(model_id: str):
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForDepthEstimation.from_pretrained(model_id).to(DEVICE).eval()
    return processor, model

@torch.no_grad()
def predict_depth_m(processor, model, image_rgb: np.ndarray) -> np.ndarray:
    pil = Image.fromarray(image_rgb)
    inputs = processor(images=pil, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    post = processor.post_process_depth_estimation(outputs, target_sizes=[(pil.height, pil.width)])
    return post[0]["predicted_depth"].cpu().float().numpy()

In [ ]:
depth_maps = {}  # (model_name, ratio) -> HxW meters

for name in ENABLED_MODELS:
    print(f"loading {name} ...")
    processor, model = load_depth_model(MODEL_ZOO[name])
    for r, img in frames.items():
        depth_maps[(name, r)] = predict_depth_m(processor, model, img)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"done: {len(depth_maps)} depth maps")

## 6. Qualitative failure analysis

For each frame: RGB next to each model's depth map. The question to ask of every panel:
**do the bicycle-stand hoops exist in the depth map, or are they absorbed into the ground
plane?** Thin near-field structures that vanish are exactly the obstacles USS reports.

In [ ]:
def show_depth_grid(ratio: float, vmax: float | None = 20.0):
    n = 1 + len(ENABLED_MODELS)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 3.2))
    axes[0].imshow(frames[ratio])
    axes[0].set_title("RGB", fontsize=9)
    for ax, name in zip(axes[1:], ENABLED_MODELS):
        d = depth_maps[(name, ratio)]
        im = ax.imshow(d, cmap="turbo", vmin=0, vmax=vmax)
        ax.set_title(f"{name}\nmin={d.min():.1f}m", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.04)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"t={ratio * video_duration_s:.1f}s", fontsize=10)
    fig.savefig(OUTPUT_DIR / f"depth_grid_r{int(ratio * 100):02d}.png", dpi=150, bbox_inches="tight")
    plt.show()

for r in FRAME_RATIOS:
    show_depth_grid(r)

In [ ]:
# Zoomed crop around the nearest obstacles in the final frame (45-degree approach:
# nearest stands sit lower-left). Adjust CROP for other sequences.
CROP = (slice(320, 640), slice(0, 480))  # rows, cols
r_last = FRAME_RATIOS[-1]

n = 1 + len(ENABLED_MODELS)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 3.4))
axes[0].imshow(frames[r_last][CROP])
axes[0].set_title("RGB crop (nearest obstacles)", fontsize=9)
for ax, name in zip(axes[1:], ENABLED_MODELS):
    d = depth_maps[(name, r_last)][CROP]
    im = ax.imshow(d, cmap="turbo")
    ax.set_title(name, fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.04)
for ax in axes:
    ax.axis("off")
fig.suptitle("Thin-structure check: do the hoops appear as depth discontinuities?", fontsize=10)
fig.savefig(OUTPUT_DIR / "thin_structure_crop.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Quantitative comparison against USS

Proxy for "predicted distance to nearest obstacle": a low percentile of depth inside a
near-field region of interest (bottom of the image, where the labeled obstacle appears
during the approach). Compared against the nearest USS-reported object distance and the
Label V2 event distance.

This is deliberately generous to the depth models — the ROI is hand-placed and the
percentile skips outliers. If they still don't track the USS curve, the failure is real.

In [ ]:
# Near-field ROI as image fractions (y0, y1, x0, x1). Bottom band, biased left for
# this sequence's 45-degree approach.
ROI_FRAC = (0.55, 1.00, 0.00, 0.70)
NEAR_PERCENTILE = 5

def roi_slice(shape, frac):
    h, w = shape[:2]
    y0, y1, x0, x1 = frac
    return slice(int(y0 * h), int(y1 * h)), slice(int(x0 * w), int(x1 * w))

records = []
for (name, r), d in depth_maps.items():
    ys, xs = roi_slice(d.shape, ROI_FRAC)
    roi = d[ys, xs]
    records.append({
        "model": name,
        "ratio": r,
        "approx_t_s": r * video_duration_s,
        "pred_near_depth_m": float(np.percentile(roi, NEAR_PERCENTILE)),
        "pred_median_depth_m": float(np.median(roi)),
    })
results = pd.DataFrame(records).sort_values(["model", "ratio"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(uss_nearest["approx_t_s"], uss_nearest["dist_m"], "k.-", lw=1, ms=3,
        label="USS nearest object (GT)")
ax.axhline(event_distance_m, color="gray", ls="--", lw=1,
           label=f"Label V2 event ({event_distance_m:.1f} m)")
for name, grp in results.groupby("model"):
    ax.plot(grp["approx_t_s"], grp["pred_near_depth_m"], "o-", label=f"{name} (p{NEAR_PERCENTILE} ROI)")
ax.set_xlabel("approx. video time [s]")
ax.set_ylabel("distance [m]")
ax.set_ylim(0, None)
ax.set_title("Predicted near-field depth vs USS ground truth")
ax.legend(fontsize=8)
fig.savefig(OUTPUT_DIR / "pred_vs_uss.png", dpi=150, bbox_inches="tight")
plt.show()

results.to_csv(OUTPUT_DIR / "summary.csv", index=False)
results

In [ ]:
# Error at the sampled frames: interpolate the USS nearest-object curve to frame times.
uss_t = uss_nearest["approx_t_s"].to_numpy()
uss_d = uss_nearest["dist_m"].to_numpy()

results["uss_dist_m"] = np.interp(results["approx_t_s"], uss_t, uss_d)
results["abs_err_m"] = (results["pred_near_depth_m"] - results["uss_dist_m"]).abs()
results["rel_err"] = results["abs_err_m"] / results["uss_dist_m"]

summary = results.groupby("model")[["abs_err_m", "rel_err"]].mean().round(3)
summary.columns = ["MAE [m]", "mean rel. error"]
results.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(f"outputs saved to {OUTPUT_DIR}")
summary

## 8. Interpretation checklist

Evidence that supports the paper's premise (any of):
- Thin obstacles (hoops) are missing or smeared into the ground plane in the depth maps.
- Predicted near-field distance does not decrease as the USS curve decreases (no tracking).
- Relative error > ~30% inside 2 m even after being generous with ROI/percentile.

Evidence *against* the premise:
- Depth Pro (which self-estimates focal length) resolves the hoops and tracks the approach —
  then the gap is scale calibration, not perception, and the paper framing must change.

**Next steps once this looks right on the sample sequence:**
1. Run over the 200 rule-ready sequences (batch script, not notebook).
2. Replace the linear cycle-to-time mapping with real MF4 timestamps (`asammdf`).
3. Stratify errors by object type, severity class, and distance bin.
4. Add per-frame USS GT to the benchmark table (Phase 2) as the supervision signal.
